In this notebook the column What is your country or origin will be cleaned, as well as the colums that will be useful for Feed diversity. In 'The country of origin column' the different variants will be made in to one uniform form. Whereas, the columns that are useful to the feed diversity, will be put in to number from 1 to 5, where 1 is the most negative and 5 the most positive. 

In [114]:
#mats
#importing packages
import numpy as np
import pandas as pd
import scipy.stats as stats
import semopy as sem

# Reading the responses dataset 
df = pd.read_excel('Responses.xlsx')
df.shape


(685, 57)

1. First there will be looked what kind of data is in the column I "What is your country or origin?".
Country of origin will be decided based on the country the respondent is born in this study, if this is not clear, the respondent will not be used.

In [115]:
#mats
#the column that will be cleaned
col = 'What is your country of origin?'

#checking the values
df[col].value_counts(dropna=False)

What is your country of origin?
Germany        125
Spain          104
Greece          68
Netherlands     54
Germany         39
              ... 
Marocco          1
Lebanon          1
Iran             1
Fryslan          1
Netherland       1
Name: count, Length: 122, dtype: int64

In [116]:
#mats
#remove all the spaces at the beginning and end
df[col] = df[col].str.strip()

#remove capitals
df[col] = df[col].str.lower()

In [117]:
#mats
#read all the different names
df[col].value_counts()

What is your country of origin?
germany        168
spain          124
greece          89
netherlands     74
nederland       37
              ... 
turkiye          1
philippines      1
marocco          1
iran             1
fryslan          1
Name: count, Length: 91, dtype: int64

In [118]:
#mats
#change datatype to string
df[col].astype(str)

#check unique and missing values
print('Unique values:', df[col].nunique)
print('Missing:', df[col].isna().sum())


Unique values: <bound method IndexOpsMixin.nunique of 0            spain
1            spain
2            spain
3            spain
4            spain
          ...     
680          ghana
681    netherlands
682    netherlands
683    netherlands
684    netherlands
Name: What is your country of origin?, Length: 685, dtype: str>
Missing: 3


In [119]:
#mats
#fixing the different written country variants/ islands of a country/ cities instead of countries
#spelling mistakes/ written in different languages

#if the country of origin is not clear the respondent will not be used e.g.: 
#“Netherlands/Greece”, “egypt/germany”, “Netherlands/ Spain”, “England/ Spain”, “Dutch,Surinamese & Indonesian”

#starting with The Netherlands variants
NL_variants = [
    'netherlands', 'the netherlands', 'nederland', 'nederlands', 'niederlande', 'holland', 'nl',
    'netherland', 'the netherlans', 'nehterlands', 'netherworld', 'dutch',
    'amsterdam', 'amsterxam', 'uithoorn', 'fryslan', 'netherlands, amsterdam', 
    'netherlands, ethnically from lebanon',
]
df[col] = df[col].replace(NL_variants, 'The Netherlands')

#count people from The Netherlands
print((df[col] == 'The Netherlands').sum())



146


In [120]:
#mats
#doing the same for the Germany variants
GER_variants = [
'germany', 'deutschland', 'german', 'getmany', 'germsny',
'i was born in germany, but my family is originally from sri lanka.',
]

df[col] = df[col].replace(GER_variants, 'Germany')

#count people from Germany
print((df[col] == 'Germany').sum())

178


In [121]:
#mats
#doing the same for the Spain variants
ES_variants = [
    'spain', 'españa',
    ]

df[col] = df[col].replace(ES_variants, 'Spain')

#count people from Spain
print((df[col] == 'Spain').sum())

132


In [122]:
#mats
#doing the same for Greece
GR_variants = [
    'greece', 'greek', 'greece.', 'creta island in greece',
    ]
df[col] = df[col].replace(GR_variants, 'Greece')

#count people from Greece
print((df[col] == 'Greece').sum())

92


In [123]:
#mats
#doing the same for Morocco
MA_variants = ['morocco', 'morrocco', 'morroco', 'marocco']
df[col] = df[col].replace(MA_variants, 'Morocco')

#doing the same for Syria
SY_variants = ['syria', 'syrien', 'سورية']
df[col] = df[col].replace(SY_variants, 'Syria')

#other spelling mistakes and translations
df[col] = df[col].replace(['suriname', 'surinam'], 'Suriname')
df[col] = df[col].replace(['turkey', 'turkiye'], 'Türkiye')
df[col] = df[col].replace(['england', 'united kingdom'], 'United Kingdom')
df[col] = df[col].replace('kazahstan', 'Kazakhstan')
df[col] = df[col].replace('irland', 'Ireland')
df[col] = df[col].replace('bulgarian', 'Bulgaria')
df[col] = df[col].replace('frankrijk', 'France')
df[col] = df[col].replace('chinese', 'China')

#adding a capital letter at the start for the rest of the data
df[col] = df[col].str.title()

#count people per country
df[col].value_counts(dropna=False)

What is your country of origin?
Germany                          178
The Netherlands                  146
Spain                            132
Greece                            92
Syria                             24
                                ... 
Suriname, Molukken, Irak           1
Iraq                               1
Dutch,Surinamese & Indonesian      1
Philippines                        1
Iran                               1
Name: count, Length: 62, dtype: int64

2. Now the columns that will be used for research on feed diversity will be put in to number from 1 to 5, where 1 is the most negative and 5 the most positive. This will be dependent per column, meaning that e.g. strongly argee can either 1 or 5. This will be motivated per column, the column used for feed diversity will be, columns R (I see repeated topics on my feed), S (I see different opinions on the same topic), T (My feed varies within my social circle), AC (I often see new ideas and topics on my feed), AQ (I rarely see posts that challenge my own political views unless I search for them), AZ (I feel like my feed shows me a good mix of topics to stay informed (e.g., news, social issues, or different opinons))

In [124]:
#mats
#converting column R, I see repeated topics on my feed
#agreeing with this statement means the feed keepsa showing the same topics, meaning a less diverse feed.
#this is why strongly agree is coded as 1 and strongly disagree is coded a 5. 
#in the paper this item belongs to Uniform Content Distribution (UCD1), repeated topics are a sign of a narrow feed.

col = df.columns[17]
df[col] = df[col].map({'Strongly agree': 1, 'Agree': 2, 'Neutral': 3, 'Disagree': 4, 'Strongly disagree': 5})
df = df.rename(columns={col: 'FD1'})

#printing values
print(df['FD1'].value_counts(dropna=False))


FD1
2.0    395
1.0    227
3.0     50
4.0      6
NaN      5
5.0      2
Name: count, dtype: int64


In [125]:
#mats
#converting column S, I see different opinions on the same topic
#seeing different opinions more often means a more diverse feed, so never is coded as 1 and always as 5.
#nobody chose rarely, so the value 2 does not appear.
#in the paper this item belongs to Uniform Content Distribution (UCD2), viewpoint diversity within topics.

col = df.columns[18]
df[col] = df[col].map({'Never': 1, 'Rarely': 2, 'Sometimes': 3, 'Often': 4, 'Always': 5})
df = df.rename(columns={col: 'FD2'})

#printing values
print(df['FD2'].value_counts(dropna=False))

FD2
3.0    300
4.0    275
5.0     76
1.0     30
NaN      4
Name: count, dtype: int64


In [126]:
#mats
#converting column T, My feed varies within my social circle
#agreeing means the feed differs from what people around you see, meaning a more diverse feed.
#this is why strongly disagree is coded as 1 and strongly agree is coded as 5.
#in the paper this item belongs to Uniform Content Distribution (UCD3), content variation across social networks.

col = df.columns[19]
df[col] = df[col].map({'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5})
df = df.rename(columns={col: 'FD3'})

#printing values
print(df['FD3'].value_counts(dropna=False))

FD3
4.0    346
3.0    193
2.0     71
5.0     65
1.0      5
NaN      5
Name: count, dtype: int64


In [127]:
#mats
#converting column AC, I often see new ideas and topics on my feed
#agreeing means the feed keeps showing new content, meaning a more diverse feed.
#this is why strongly disagree is coded as 1 and strongly agree is coded as 5.
#in the paper this item belongs to Reinforced Exisiting Views and Preferences (REVP5), seeing new or challenging information

col = df.columns[28]
df[col] = df[col].map({'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5})
df = df.rename(columns={col: 'FD4'})

#printing values
print(df['FD4'].value_counts(dropna=False))

FD4
4.0    317
3.0    241
2.0     84
5.0     31
1.0      7
NaN      5
Name: count, dtype: int64


In [128]:
#mats
#converting column AQ, I rarely see posts that challenge my own political views unless I search for them
#agreeing means the feed mostly confirms your own views, meaning a less diverse feed.
#this is why strongly agree is coded as 1 and strongly disagree is coded as 5 (reverse coded).
#in the paper this item belongs to Diverse Viewpoints (DVP4), the limitation of encountering diverse viewpoints.

col = df.columns[42]
df[col] = df[col].map({'Strongly agree': 1, 'Agree': 2, 'Neutral': 3, 'Disagree': 4, 'Strongly disagree': 5})
df = df.rename(columns={col: 'FD5'})

#printing values
print(df['FD5'].value_counts(dropna=False))

FD5
2.0    243
3.0    201
4.0    156
1.0     59
5.0     17
NaN      9
Name: count, dtype: int64


In [129]:
#mats
#converting column AZ, I feel like my feed shows me a good mix of topics to stay informed
#agreeing means the feed shows a variety of topics and opinions, meaning a more diverse feed.
#this is why strongly disagree is coded as 1 and strongly agree is coded as 5.
#in the paper this item belongs to Opportunity & Need (ON1); the variety of the feed.

col = df.columns[51]
df[col] = df[col].map({'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5})
df = df.rename(columns={col: 'FD6'})

#printing values
print(df['FD6'].value_counts(dropna=False))

FD6
4.0    283
3.0    191
2.0    125
5.0     56
1.0     21
NaN      9
Name: count, dtype: int64


FD1 = column R: I see repeated topics on my feed

FD2 = column S: I see different opinions on the same topic

FD3 = column T: My feed varies within my social circle

FD4 = column AC: I often see new ideas and topics on my feed

FD5 = column AQ: I rarely see posts that challenge my own political views unless I search for them

FD6 = column AZ: I feel like my feed shows me a good mix of topics to stay informed

PF1 = column BC: I trust the algorithm to recommend me content fairly

PF2 = column BE: converting column BE, I think social media is fair2

3. markdown why we look into EV too

In [ ]:
#mats
#converting column AE, My feed does not show information that others could be seeing
#agreeing means the feed hides information that others see (a filter bubble), meaning more biased content.
#this is why strongly agree is coded as 1 and strongly disagree is coded as 5 (reverse coded).
#in the paper this item belongs to Biased Content (BC1), filter bubbles. The column is renamed to BC1.

col = df.columns[30]
df[col] = df[col].map({'Strongly agree': 1, 'Agree': 2, 'Neutral': 3, 'Disagree': 4, 'Strongly disagree': 5})
df = df.rename(columns={col: 'BC1'})

#printing values
print(df['BC1'].value_counts(dropna=False))

In [ ]:
#mats
#converting column AF, I feel like my content gets the same number of views as content from others
#agreeing means the respondent feels content gets equal visibility.
#this is why strongly disagree is coded as 1 and strongly agree is coded as 5.
#in the paper this item belongs to Biased Content (BC2), visibility bias. The column is renamed to EV1 (equal visibility)

col = df.columns[31]
df[col] = df[col].map({'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5})
df = df.rename(columns={col: 'BC2'})

#printing values
print(df['BC2'].value_counts(dropna=False))

EV1
3.0    296
2.0    183
4.0    119
1.0     65
NaN     14
5.0      8
Name: count, dtype: int64


In [ ]:
#mats
#converting column AG, I feel like my content gets the same number of views as content from others with a different background
#agreeing means the respondent feels people from all backgrounds get equal visibility.
#this is why strongly disagree is coded as 1 and strongly agree is coded as 5.
#in the paper this item belongs to Biased Content (BC3), visibility bias. The column is renamed to EV2.

col = df.columns[32]
df[col] = df[col].map({'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5})
df = df.rename(columns={col: 'BC3'})

#printing values
print(df['BC3'].value_counts(dropna=False))

EV2
3.0    345
4.0    135
2.0    135
1.0     44
NaN     15
5.0     11
Name: count, dtype: int64


In [ ]:
#mats
#converting column AH, I think content from certain groups of people is prioritized over other groups
#agreeing means the respondent feels some groups get more visibility, meaning less equal visibility.
#this is why strongly agree is coded as 1 and strongly disagree is coded as 5 (reverse coded).
#in the paper this item belongs to Biased Content (BC4), algorithmic bias. The column is renamed to EV3.

col = df.columns[33]
df[col] = df[col].map({'Strongly agree': 1, 'Agree': 2, 'Neutral': 3, 'Disagree': 4, 'Strongly disagree': 5})
df = df.rename(columns={col: 'BC4'})

#printing values
print(df['BC4'].value_counts(dropna=False))

EV3
2.0    332
3.0    153
1.0    121
4.0     64
5.0      8
NaN      7
Name: count, dtype: int64


In [ ]:
#mats
#converting column AI, I am informed that my feed is shaped by personal information like my age, gender or location
#agreeing means the platform is transparent about how the feed is personalised, meaning less biased content.
#this is why strongly disagree is coded as 1 and strongly agree is coded as 5.
#in the paper this item belongs to Biased Content (BC5), transparency. The column is renamed to BC5.

col = df.columns[34]
df[col] = df[col].map({'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5})
df = df.rename(columns={col: 'BC5'})

#printing values
print(df['BC5'].value_counts(dropna=False))

BC1 = column AE: My feed does not show information that others could be seeing 
BC2 = column AF: I feel like my content gets the same number of views as content from others

BC3 = column AG: I feel like my content gets the same number of views as content from others with a different background

BC4 = column AH: I think content from certain groups of people is prioritized over other groups 

BC5 = column AI: I am informed that my feed is shaped by personal information like my age, gender or location

In [135]:
df.to_excel('cleaned_data_mats.xlsx', index=False)